# Palace Field Visualization

This notebook extends the CPW demo to save and visualize 3D electromagnetic
fields from a Palace driven simulation.

Palace can export full E/H fields as ParaView `.vtu` files at specific
frequencies. After the cloud run completes, we load them with
[PyVista](https://docs.pyvista.org) for interactive 3D visualization.

**Requirements:**
- IHP PDK: `uv pip install ihp-gdsfactory`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation
- PyVista + meshio: `uv pip install pyvista meshio` (for field visualization)

## 1. Define the CPW structure

In [ ]:
import gdsfactory as gf
from ihp import LAYER, PDK

PDK.activate()


@gf.cell
def gsg_electrode(
    length: float = 800,
    s_width: float = 20,
    g_width: float = 40,
    gap_width: float = 15,
    layer=LAYER.TopMetal2drawing,
) -> gf.Component:
    c = gf.Component()
    r1 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r1.move((0, (g_width + s_width) / 2 + gap_width))
    _r2 = c << gf.c.rectangle((length, s_width), centered=True, layer=layer)
    r3 = c << gf.c.rectangle((length, g_width), centered=True, layer=layer)
    r3.move((0, -(g_width + s_width) / 2 - gap_width))
    c.add_port(
        name="o1",
        center=(-length / 2, 0),
        width=s_width,
        orientation=0,
        port_type="electrical",
        layer=layer,
    )
    c.add_port(
        name="o2",
        center=(length / 2, 0),
        width=s_width,
        orientation=180,
        port_type="electrical",
        layer=layer,
    )
    return c


c = gsg_electrode()
cc = c.copy()
cc.draw_ports()
cc

## 2. Configure simulation with field saving

Two ways to save fields:
- **`save_step=N`** — save fields every N-th frequency step
- **`save_fields_at=[f1, f2, ...]`** — save fields at specific frequencies (Hz)

We'll save at 10 GHz and 50 GHz to compare low- and mid-frequency field patterns.

Note: Palace requires `Save` frequencies to exactly match the sample grid.
gsim automatically snaps to the nearest sample point and warns if the shift
is significant.

In [ ]:
from gsim.palace import DrivenSim

sim = DrivenSim()
sim.set_output_dir("./palace-sim-cpw-fields")
sim.set_geometry(c)
sim.set_stack(substrate_thickness=2.0, air_above=300.0)

sim.add_cpw_port("o1", layer="topmetal2", s_width=20, gap_width=15)
sim.add_cpw_port("o2", layer="topmetal2", s_width=20, gap_width=15)

# Save fields at 10 GHz and 50 GHz
sim.set_driven(
    fmin=1e9,
    fmax=100e9,
    num_points=300,
    save_fields_at=[10e9, 50e9],
)

print(sim.validate_config())

In [ ]:
sim.mesh(preset="default", planar_conductors=False)

## 3. Run on cloud

In [ ]:
results = sim.run()

## 4. Plot S-parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(results["port-S.csv"])
df.columns = df.columns.str.strip()

freq = df["f (GHz)"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6))
ax1.plot(freq, df["|S[1][1]| (dB)"], label="S11")
ax1.plot(freq, df["|S[2][1]| (dB)"], label="S21")
ax1.set_xlabel("Frequency (GHz)")
ax1.set_ylabel("Magnitude (dB)")
ax1.set_title("S-Parameters")
ax1.axvline(10, color="gray", ls="--", alpha=0.5, label="saved fields")
ax1.axvline(50, color="gray", ls="--", alpha=0.5)
ax1.legend()
ax1.grid(True)

ax2.plot(freq, df["arg(S[1][1]) (deg.)"], label="S11")
ax2.plot(freq, df["arg(S[2][1]) (deg.)"], label="S21")
ax2.set_xlabel("Frequency (GHz)")
ax2.set_ylabel("Phase (deg)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()

## 5. Field visualization utilities

Adapted from [palace-server](https://github.com/doplaydo/palace-server)
postprocessing module. These functions use meshio to read the `.msh` mesh,
filter out vacuum regions, and interpolate the volumetric field data onto
the conductor/dielectric surfaces for clean rendering.

In [ ]:
from __future__ import annotations

from pathlib import Path
from xml.etree import ElementTree

import meshio
import numpy as np
import pyvista as pv


def _build_surface(msh, ignore_groups=None):
    """Extract surface mesh from .msh, filtering out air/vacuum/port groups."""
    if ignore_groups is None:
        ignore_groups = [
            name
            for name in msh.field_data
            if "air" in name.lower()
            or "vacuum" in name.lower()
            or (name.lower().startswith("p") and "_e" in name.lower())
        ]
    tags_to_ignore = {v[0] for k, v in msh.field_data.items() if k in ignore_groups}

    tri_data = next(c.data for c in msh.cells if c.type == "triangle")
    tri_tags = msh.cell_data_dict["gmsh:physical"]["triangle"]
    keep = [tri for tri, tag in zip(tri_data, tri_tags) if tag not in tags_to_ignore]

    if not keep:
        raise ValueError(
            f"No triangles left after filtering {ignore_groups}. "
            f"Available groups: {list(msh.field_data.keys())}"
        )
    faces = np.insert(np.array(keep), 0, 3, axis=1)
    return pv.PolyData(msh.points, faces=faces)


def plot_boundary_field(
    field_file: str | Path,
    field_name: str = "J_s_real",
    component: int | None = None,
    cmap: str = "turbo",
    log_scale: bool = True,
    show_edges: bool = True,
    opacity: float = 1.0,
    clim: tuple[float, float] | None = None,
    off_screen: bool = False,
) -> pv.Plotter:
    """Plot a boundary field directly from the Palace boundary .pvtu."""
    grid = pv.read(str(field_file))

    if field_name not in grid.array_names:
        raise ValueError(
            f"Field '{field_name}' not found. Available: {grid.array_names}"
        )

    arr = grid.point_data[field_name]
    if component is None and arr.ndim == 2:
        grid["_mag"] = np.linalg.norm(arr, axis=1)
        scalars = "_mag"
    else:
        scalars = field_name

    pl = pv.Plotter(off_screen=off_screen, window_size=(1280, 720))
    pl.set_background("paraview")

    mesh_kwargs = dict(
        scalars=scalars,
        cmap=cmap,
        log_scale=log_scale,
        smooth_shading=True,
        show_edges=show_edges,
        edge_color="gray",
        opacity=opacity,
        show_scalar_bar=True,
        scalar_bar_args={"title": field_name},
    )
    if component is not None:
        mesh_kwargs["component"] = component
    if clim is not None:
        mesh_kwargs["clim"] = clim

    pl.add_mesh(grid, **mesh_kwargs)
    pl.add_axes()
    pl.show_grid()
    pl.camera_position = "iso"
    pl.camera.zoom(1.2)
    pl.enable_anti_aliasing("fxaa")
    pl.enable_lightkit()
    return pl


def plot_field(
    msh_file: str | Path,
    field_file: str | Path,
    field_name: str = "E_real",
    component: int | None = None,
    ignore_groups: list[str] | None = None,
    cmap: str = "turbo",
    opacity: float = 1.0,
    show_edges: bool = False,
    off_screen: bool = False,
) -> pv.Plotter:
    """Plot a Palace field on the mesh surface (excluding vacuum).

    For boundary-native fields (J_s, Q_s), prefer plot_boundary_field().
    """
    msh = meshio.read(str(msh_file))
    grid = pv.read(str(field_file))

    if field_name not in grid.array_names:
        raise ValueError(
            f"Field '{field_name}' not found. Available: {grid.array_names}"
        )

    surf = _build_surface(msh, ignore_groups)
    surf = surf.interpolate(
        grid,
        sharpness=2.0,
        radius=1e-6,
        strategy="closest_point",
        pass_point_data=True,
    )

    pl = pv.Plotter(off_screen=off_screen, window_size=(1280, 720))
    pl.set_background("paraview")
    pl.add_mesh(
        surf,
        scalars=field_name,
        component=component,
        cmap=cmap,
        smooth_shading=True,
        opacity=opacity,
        show_edges=show_edges,
    )
    pl.add_axes()
    pl.show_grid()
    pl.camera_position = "iso"
    pl.camera.zoom(1.2)
    pl.enable_anti_aliasing("fxaa")
    pl.enable_lightkit()
    return pl


def plot_field_slice(
    field_file: str | Path,
    field_name: str = "E_real",
    component: int | None = None,
    normal: str = "z",
    origin: tuple[float, ...] | None = None,
    cmap: str = "turbo",
    log_scale: bool = True,
    clim: tuple[float, float] | None = None,
    off_screen: bool = False,
) -> pv.Plotter:
    """Plot a cross-section slice through the volumetric field data.

    For vector fields, plots |field| magnitude by default.
    Pass component=0/1/2 for X/Y/Z components.
    """
    grid = pv.read(str(field_file))

    if field_name not in grid.array_names:
        raise ValueError(
            f"Field '{field_name}' not found. Available: {grid.array_names}"
        )

    # For vector fields: compute magnitude or extract component
    arr = grid.point_data[field_name]
    if arr.ndim == 2:
        if component is not None:
            grid["_scalar"] = arr[:, component]
            labels = {0: "x", 1: "y", 2: "z"}
            title = f"{field_name}_{labels.get(component, component)}"
        else:
            grid["_scalar"] = np.linalg.norm(arr, axis=1)
            title = f"|{field_name}|"
        scalars = "_scalar"
    else:
        scalars = field_name
        title = field_name

    if origin is None:
        b = grid.bounds
        origin = ((b[0] + b[1]) / 2, (b[2] + b[3]) / 2, (b[4] + b[5]) / 2)

    sliced = grid.slice(normal=normal, origin=origin)

    pl = pv.Plotter(off_screen=off_screen, window_size=(1280, 720))
    pl.set_background("paraview")

    mesh_kwargs = dict(
        scalars=scalars,
        cmap=cmap,
        log_scale=log_scale,
        smooth_shading=True,
        show_scalar_bar=True,
        scalar_bar_args={"title": title},
    )
    if clim is not None:
        mesh_kwargs["clim"] = clim

    pl.add_mesh(sliced, **mesh_kwargs)
    pl.add_axes()
    view = {"x": "yz", "y": "xz", "z": "xy"}
    getattr(pl, f"view_{view.get(normal, 'xy')}")()
    pl.enable_anti_aliasing("fxaa")
    return pl


def read_pvd(pvd_file: str | Path) -> list[tuple[float, Path]]:
    """Parse a .pvd file to get (timestep/freq, pvtu_path) pairs."""
    pvd_path = Path(pvd_file)
    tree = ElementTree.parse(pvd_path)
    entries = []
    for ds in tree.iter("DataSet"):
        t = float(ds.attrib["timestep"])
        f = pvd_path.parent / ds.attrib["file"]
        if f.exists():
            entries.append((t, f))
    return sorted(entries, key=lambda x: x[0])


def find_field_file(output_dir: str | Path, freq_ghz: float) -> Path | None:
    """Find the .pvtu file closest to a given frequency."""
    output_dir = Path(output_dir)

    for pvd_file in sorted(output_dir.rglob("*.pvd")):
        entries = read_pvd(pvd_file)
        if entries:
            best = min(entries, key=lambda e: abs(e[0] - freq_ghz))
            return best[1]

    vtu_files = sorted(output_dir.rglob("*.vtu"))
    if not vtu_files:
        return None
    import re

    best_file, best_dist = None, float("inf")
    for f in vtu_files:
        match = re.search(r"([\d.]+)\.vtu$", f.name)
        if match:
            file_freq = float(match.group(1))
            dist = abs(file_freq - freq_ghz)
            if dist < best_dist:
                best_file, best_dist = f, dist
    return best_file or vtu_files[0]


print("Viz utilities loaded.")

## 6. Visualize fields

Palace saves two kinds of field data:

- **Volume** (`driven/`) — full 3D fields, best for cross-section slices
- **Boundary** (`driven_boundary/`) — fields on surfaces only, includes surface current and charge

| Field | Type | Description |
|-------|------|-------------|
| `E_real`, `E_imag` | vector | Electric field |
| `B_real`, `B_imag` | vector | Magnetic flux density |
| `S` | vector | Poynting vector (power flow) |
| `U_e`, `U_m` | scalar | Electric / magnetic energy density |
| `J_s_real`, `J_s_imag` | vector | Surface current density (boundary only) |
| `Q_s_real`, `Q_s_imag` | scalar | Surface charge density (boundary only) |

Component: `0`=X, `1`=Y, `2`=Z, `None`=magnitude

### List available field files

In [ ]:
msh_file = Path("palace-sim-cpw-fields") / "palace.msh"
results_dir = Path("sim-data-palace-a2a1bbd0")

# Volume fields (for cross-section slices)
vol_dir = results_dir / "results/palace/paraview/driven/excitation_1"
# Boundary fields (for surface current, charge)
bnd_dir = results_dir / "results/palace/paraview/driven_boundary/excitation_1"


def find_pvtu(pvd_dir, freq_ghz):
    """Find .pvtu closest to freq_ghz from a .pvd in the given directory."""
    for pvd in sorted(pvd_dir.glob("*.pvd")):
        entries = read_pvd(pvd)
        if entries:
            best = min(entries, key=lambda e: abs(e[0] - freq_ghz))
            return best[1]
    return None


# Check what's available
print("Volume .pvd entries:")
for t, f in read_pvd(vol_dir / "excitation_1.pvd"):
    print(f"  {t:.3f} GHz -> {f.name}")

print("\nBoundary .pvd entries:")
for t, f in read_pvd(bnd_dir / "excitation_1.pvd"):
    print(f"  {t:.3f} GHz -> {f.name}")

# Inspect available arrays
vol_file = find_pvtu(vol_dir, 10.0)
bnd_file = find_pvtu(bnd_dir, 10.0)
print(f"\nVolume arrays:   {pv.read(str(vol_file)).array_names}")
print(f"Boundary arrays: {pv.read(str(bnd_file)).array_names}")

### Surface current density on conductors

In [ ]:
# Surface current density as colored wireframe — clip air above conductors
bnd_10 = find_pvtu(bnd_dir, 10.0)

grid = pv.read(str(bnd_10))
grid["J_s_mag"] = np.linalg.norm(grid.point_data["J_s_real"], axis=1)

# Clip: remove everything above the conductor layer
clipped = grid.clip(normal="z", origin=(0, 0, 17), invert=True)

pl = pv.Plotter(window_size=(1280, 720))
pl.set_background("white")
pl.add_mesh(
    clipped,
    scalars="J_s_mag",
    cmap="turbo",
    style="wireframe",
    line_width=1,
    clim=[1, 1e4],
    log_scale=True,
    show_scalar_bar=True,
    scalar_bar_args={"title": "|J_s| (A/m)"},
)
pl.add_axes()
pl.camera_position = "iso"
pl.camera.zoom(1.5)
pl.enable_anti_aliasing("fxaa")
pl.show()

### Surface charge density

In [ ]:
# Surface charge density as colored wireframe — clip air above conductors
grid = pv.read(str(bnd_10))
clipped = grid.clip(normal="z", origin=(0, 0, 17), invert=True)

pl = pv.Plotter(window_size=(1280, 720))
pl.set_background("white")
pl.add_mesh(
    clipped,
    scalars="Q_s_real",
    cmap="RdBu",
    style="wireframe",
    line_width=2,
    show_scalar_bar=True,
    scalar_bar_args={"title": "Q_s (C/m²)"},
)
pl.add_axes()
pl.camera_position = "iso"
pl.camera.zoom(1.5)
pl.enable_anti_aliasing("fxaa")
pl.show()

### Cross-section slices (volume data)

`plot_field_slice` cuts directly through the 3D volumetric field —
much smoother than surface interpolation.

In [ ]:
# E-field magnitude — XY slice (top-down view)
vol_10 = find_pvtu(vol_dir, 10.0)
pl = plot_field_slice(vol_10, field_name="E_real", normal="z")
pl.show()

In [ ]:
# E-field — YZ cross-section through the CPW gap
pl = plot_field_slice(vol_10, field_name="E_real", normal="x")
pl.show()

In [ ]:
### Energy density and power flow

# Electric energy density — where is E-field energy concentrated?
pl = plot_field_slice(vol_10, field_name="U_e", normal="z")
pl.show()

In [ ]:
# Poynting vector (power flow) — XY slice
pl = plot_field_slice(vol_10, field_name="S", normal="z")
pl.show()

### Magnetic field cross-section

In [ ]:
# B-field — YZ cross-section (see field wrapping around the conductor)
pl = plot_field_slice(vol_10, field_name="B_real", normal="x")
pl.show()

## Notes

- **`plot_field()`** — surface rendering using boundary data. Best for `J_s` (current) and `Q_s` (charge).
- **`plot_field_slice()`** — cross-section through 3D volume data. Best for `E`, `B`, `U_e`, `U_m`, `S`.
- `save_step=N` saves at every N-th step (many files). `save_fields_at=[f1, f2]` is more targeted.
- **ParaView** can also open the `.pvtu` files for streamlines, clip planes, and more.
- Viz functions adapted from [palace-server](https://github.com/doplaydo/palace-server).